# Sinhala Document OCR - Colab Baseline Pipeline

This notebook runs an **end-to-end baseline** for the Sinhala line-recognition
CRNN (CNN + BiLSTM + CTC) on a Colab **GPU** runtime. Each step maps directly onto
the project's `src/` modules so you can read the code alongside the notebook.

**Pipeline**
1. Clone the repository and install dependencies.
2. Install a Sinhala-capable font (Colab is Linux, so Windows fonts are absent).
3. Generate a synthetic Sinhala line dataset with a seeded train/val/test split.
4. Train the CRNN for a few epochs, logging train loss and validation CER, saving the best checkpoint.
5. Evaluate on the held-out test set (mean CER / WER) with qualitative examples.
6. Run an inference demo on a single line image.

> **Tip:** enable the GPU via *Runtime -> Change runtime type -> Hardware accelerator: GPU*.

## 1. Clone the project and install dependencies

We clone the public repository, `cd` into it, and install the Python requirements.
The notebook is **OS-aware**: it detects whether it is running inside Colab (Linux).

In [ ]:
import os

REPO_URL = "https://github.com/Hellsgate96/sinhala-document-ocr-2026.git"
REPO_DIR = "sinhala-document-ocr-2026"

# Detect Colab (Linux) vs. a local run.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

if IN_COLAB and not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
print("Working directory:", os.getcwd())

In [ ]:
# torch / numpy / Pillow are usually preinstalled in Colab; this is quick if so.
!pip -q install -r requirements.txt
!pip -q install editdistance tqdm pyyaml

## 2. Install a Sinhala-capable font

The synthetic generator renders text with a TrueType font. Colab runs on Linux, so
the Windows default (`Nirmala.ttc`) does not exist. We install **Noto Sans Sinhala**
from the `fonts-noto-core` apt package, with a direct download as a robust fallback,
then locate the resulting `.ttf` on disk.

In [ ]:
import glob

FONT_PATH = None
if IN_COLAB:
    # 1) Noto Sans Sinhala ships inside the fonts-noto-core apt package.
    !apt-get -qq update > /dev/null 2>&1
    !apt-get -qq install -y fonts-noto-core > /dev/null 2>&1 || true
    # 2) Robust fallback: download a static Noto Sans Sinhala TTF.
    os.makedirs("fonts", exist_ok=True)
    if not glob.glob("fonts/*Sinhala*.ttf"):
        !wget -q -O fonts/NotoSansSinhala-Regular.ttf "https://github.com/notofonts/sinhala/raw/main/fonts/NotoSansSinhala/hinted/ttf/NotoSansSinhala-Regular.ttf" || true

# Search common locations for a Sinhala-capable font file.
candidates = (
    glob.glob("/usr/share/fonts/**/*Sinhala*.ttf", recursive=True)
    + glob.glob("fonts/*Sinhala*.ttf")
    + ["C:/Windows/Fonts/Nirmala.ttc"]  # local Windows fallback
)
candidates = [c for c in candidates if os.path.isfile(c)]
assert candidates, "No Sinhala font found - re-run this cell or `apt-get install fonts-noto-core`."
FONT_PATH = candidates[0]
print("Using Sinhala font:", FONT_PATH)

## 3. Load the configuration

`configs/default.yaml` is the single source of truth for paths, image geometry,
synthetic-data settings and training hyper-parameters. We load it, point the
generator at the Colab font, and choose baseline dataset / training sizes.

In [ ]:
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.utils.common import load_config, configure_stdout_utf8, get_logger
configure_stdout_utf8()

cfg = load_config("configs/default.yaml")

# OS-aware: use the font we just installed.
cfg["synthetic"]["fonts"] = [FONT_PATH]

# Baseline sizes - tune these for longer runs.
NUM_SAMPLES = 10000     # first run: ~8k-12k lines
EPOCHS = 15
BATCH_SIZE = 64
OUT_DIR = cfg["paths"]["synthetic_dir"]   # data/synthetic

print("image height :", cfg["image"]["height"], "| max_width:", cfg["image"]["max_width"])
print("charset path :", cfg["paths"]["charset_path"])
print("output dir   :", OUT_DIR)

## 4. Generate the synthetic dataset

`generate(...)` renders `NUM_SAMPLES` augmented line images, then writes
`all_labels.txt` plus the per-split `train_labels.txt` / `val_labels.txt` /
`test_labels.txt` (each row is `relative_image_path<TAB>transcript`). Vocabulary is
drawn from `sample_words.txt` and the form-field list `form_vocab.txt`. A `tqdm`
progress bar shows rendering progress.

In [ ]:
from src.data.synthetic_generator import generate, load_word_lists

logger = get_logger("generate")
words = load_word_lists([cfg["paths"]["word_list"], cfg["paths"].get("form_vocab")],
                        warn=logger.warning)
print("Vocabulary entries:", len(words))

counts = generate(
    out_dir=OUT_DIR,
    num_samples=NUM_SAMPLES,
    font_paths=[FONT_PATH],
    font_sizes=cfg["synthetic"]["font_sizes"],
    words=words,
    min_words=cfg["synthetic"]["min_words"],
    max_words=cfg["synthetic"]["max_words"],
    augment=cfg["synthetic"]["augment"],
    split=cfg["synthetic"]["split"],
    seed=cfg["project"]["seed"],
    logger=logger,
    numeric_ratio=cfg["synthetic"]["numeric_ratio"],
    mixed_ratio=cfg["synthetic"]["mixed_ratio"],
)
print("Split counts:", counts)

### Preview a few generated lines

We register the Sinhala font with matplotlib so the transcripts render correctly in
the titles.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
from src.data.dataset import read_labels

try:
    if FONT_PATH.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(FONT_PATH)
        plt.rcParams["font.family"] = fm.FontProperties(fname=FONT_PATH).get_name()
except Exception as e:
    print("(font registration skipped:", e, ")")

rows = read_labels(os.path.join(OUT_DIR, "train_labels.txt"))
fig, axes = plt.subplots(5, 1, figsize=(10, 7))
for ax, (rel, text) in zip(axes, rows[:5]):
    ax.imshow(Image.open(os.path.join(OUT_DIR, rel)))
    ax.set_title(text, fontsize=12)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train the CRNN baseline

`src/recognition/train.py` builds the shared `Charset` (saved next to the
checkpoints), trains with `torch.nn.CTCLoss`, logs **train loss** and **validation
CER** each epoch, and saves `crnn_best.pth` (lowest val CER) + `crnn_last.pth` to
`models/`. We pass config overrides on the command line.

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)

cmd = (
    f"python -m src.recognition.train --config configs/default.yaml "
    f"paths.synthetic_dir={OUT_DIR} train.epochs={EPOCHS} "
    f"train.batch_size={BATCH_SIZE} train.device={DEVICE}"
)
print(cmd)
!{cmd}

## 6. Evaluate on the held-out test set

We load the best checkpoint and the saved charset, build a test `DataLoader`, and
compute corpus **CER** and **WER** with `evaluate_model(...)`.

In [ ]:
from src.charset import Charset
from src.data.dataset import build_dataloader
from src.recognition.model import build_crnn
from src.evaluation.metrics import evaluate_model
from src.utils.common import load_checkpoint, get_device

charset = Charset.load(cfg["paths"]["charset_path"])
device = get_device(DEVICE)
model = build_crnn(charset.num_classes, cfg.get("model"),
                   in_channels=cfg["image"]["channels"]).to(device)
load_checkpoint("models/crnn_best.pth", model, map_location=str(device))
model.eval()

test_loader = build_dataloader(
    os.path.join(OUT_DIR, "test_labels.txt"), charset,
    batch_size=cfg["train"]["batch_size"], height=cfg["image"]["height"],
    max_width=cfg["image"]["max_width"], channels=cfg["image"]["channels"],
    shuffle=False, num_workers=2)

report = evaluate_model(model, test_loader, charset, device=device, measure_cpu_time=False)
print(f"TEST  samples={report['num_samples']}  CER={report['cer']:.4f}  WER={report['wer']:.4f}")

### Qualitative predictions (image + ground truth + prediction)

In [ ]:
test_rows = read_labels(os.path.join(OUT_DIR, "test_labels.txt"))
n = min(5, len(test_rows))
fig, axes = plt.subplots(n, 1, figsize=(10, 1.8 * n))
if n == 1:
    axes = [axes]
for ax, (rel, _), s in zip(axes, test_rows[:n], report["per_sample"][:n]):
    ax.imshow(Image.open(os.path.join(OUT_DIR, rel)))
    ax.set_title(f"GT: {s['ref']}    PRED: {s['hyp']}    (CER={s['cer']:.2f})", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Inference demo on a single image

`predict_image(...)` from `src/recognition/predict.py` performs greedy CTC decoding
on one line image and returns the decoded Sinhala string.

In [ ]:
from src.recognition.predict import predict_image

sample_rel, sample_gt = read_labels(os.path.join(OUT_DIR, "test_labels.txt"))[0]
sample_path = os.path.join(OUT_DIR, sample_rel)
pred = predict_image(model, charset, sample_path,
                     cfg["image"]["height"], cfg["image"]["max_width"],
                     cfg["image"]["channels"], device)
print("Image      :", sample_path)
print("Ground truth:", sample_gt)
print("Prediction :", pred)
Image.open(sample_path)

## Optional: persist the model to Google Drive

Uncomment to mount Drive and copy the best checkpoint + charset so they survive the
Colab session.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# os.makedirs('/content/drive/MyDrive/sinhala_ocr', exist_ok=True)
# shutil.copy('models/crnn_best.pth', '/content/drive/MyDrive/sinhala_ocr/')
# shutil.copy('models/charset.json', '/content/drive/MyDrive/sinhala_ocr/')
print("Uncomment the lines above to save the model + charset to Google Drive.")